# **Security incidents**

## **Import libraries**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
sns.set_style("whitegrid")

In [2]:
sns.set_palette('viridis')
sns.color_palette(palette='viridis')

[(0.275191, 0.194905, 0.496005),
 (0.212395, 0.359683, 0.55171),
 (0.153364, 0.497, 0.557724),
 (0.122312, 0.633153, 0.530398),
 (0.288921, 0.758394, 0.428426),
 (0.626579, 0.854645, 0.223353)]

In [3]:
plt.set_loglevel('warning')

## **Load Dataset**

In [4]:
df = pd.read_csv("data.csv")

In [5]:
df.head()

,Timestamp,Source IP Address,Destination IP Address,Source Port,Destination Port,Protocol,Packet Length,Packet Type,Traffic Type,Payload Data,...,Action Taken,Severity Level,User Information,Device Information,Network Segment,Geo-location Data,Proxy Information,Firewall Logs,IDS/IPS Alerts,Log Source
0,2023-05-30 06:33:58,103.216.15.12,84.9.164.252,31225,17616,ICMP,503,Data,HTTP,Qui natus odio asperiores nam. Optio nobis ius...,...,Logged,Low,Reyansh Dugal,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment A,"Jamshedpur, Sikkim",150.9.97.135,Log Data,NaN,Server
1,2020-08-26 07:08:30,78.199.217.198,66.191.137.154,17245,48166,ICMP,1174,Data,HTTP,Aperiam quos modi officiis veritatis rem. Omni...,...,Blocked,Low,Sumer Rana,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment B,"Bilaspur, Nagaland",NaN,Log Data,NaN,Firewall
2,2022-11-13 08:23:25,63.79.210.48,198.219.82.17,16811,53600,UDP,306,Control,HTTP,Perferendis sapiente vitae soluta. Hic delectu...,...,Ignored,Low,Himmat Karpe,Mozilla/5.0 (compatible; MSIE 9.0; Windows NT ...,Segment C,"Bokaro, Rajasthan",114.133.48.179,Log Data,Alert Data,Firewall
3,2023-07-02 10:38:46,163.42.196.10,101.228.192.255,20018,32534,UDP,385,Data,HTTP,Totam maxime beatae expedita explicabo porro l...,...,Blocked,Medium,Fateh Kibe,Mozilla/5.0 (Macintosh; PPC Mac OS X 10_11_5; ...,Segment B,"Jaunpur, Rajasthan",NaN,NaN,Alert Data,Firewall
4,2023-07-16 13:11:07,71.166.185.76,189.243.174.238,6131,26646,TCP,1462,Data,DNS,Odit nesciunt dolorem nisi iste iusto. Animi v...,...,Blocked,Low,Dhanush Chad,Mozilla/5.0 (compatible; MSIE 5.0; Windows NT ...,Segment C,"Anantapur, Tripura",149.6.110.119,NaN,Alert Data,Firewall


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Timestamp               40000 non-null  object 
 1   Source IP Address       40000 non-null  object 
 2   Destination IP Address  40000 non-null  object 
 3   Source Port             40000 non-null  int64  
 4   Destination Port        40000 non-null  int64  
 5   Protocol                40000 non-null  object 
 6   Packet Length           40000 non-null  int64  
 7   Packet Type             40000 non-null  object 
 8   Traffic Type            40000 non-null  object 
 9   Payload Data            40000 non-null  object 
 10  Malware Indicators      20000 non-null  object 
 11  Anomaly Scores          40000 non-null  float64
 12  Alerts/Warnings         19933 non-null  object 
 13  Attack Type             40000 non-null  object 
 14  Attack Signature        40000 non-null

## **Data Preprocessing**

### **Tranform**

Transform Timestamp into a viable format

In [7]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

### **Drop**

Drop unuseful(found in EDA) features.

In [8]:
feat_to_drop = ['Source IP Address', 'Destination IP Address',
       'Source Port', 'Destination Port', 'Payload Data', 'User Information']

In [9]:
df.drop(feat_to_drop, axis=1, inplace=True)

### **Handle Missing**

In [10]:
missing_features = ['Malware Indicators', 'Alerts/Warnings', 'Proxy Information',
                    'Firewall Logs', 'IDS/IPS Alerts']

In [11]:
df[missing_features] = df[missing_features].notna().astype(int)

As it was shown in EDA, the main pattern in this data is either it is present or not.

## **Feature Engineering**

### **Encoding**

In [12]:
df.columns

Index(['Timestamp', 'Protocol', 'Packet Length', 'Packet Type', 'Traffic Type',
       'Malware Indicators', 'Anomaly Scores', 'Alerts/Warnings',
       'Attack Type', 'Attack Signature', 'Action Taken', 'Severity Level',
       'Device Information', 'Network Segment', 'Geo-location Data',
       'Proxy Information', 'Firewall Logs', 'IDS/IPS Alerts', 'Log Source'],
      dtype='object')

Low-cardinality features should be one-hot encoded

In [13]:
low_card_cat_features = df.loc[:,df.nunique() < 10].select_dtypes(exclude='number').columns.tolist()

In [14]:
low_card_cat_features

['Protocol',
 'Packet Type',
 'Traffic Type',
 'Attack Type',
 'Attack Signature',
 'Action Taken',
 'Severity Level',
 'Network Segment',
 'Log Source']

In [ ]:
low_card_cat_features.remove('Severity Level')

In [15]:
dummies = pd.get_dummies(df[low_card_cat_features])
df = pd.concat([df.drop(low_card_cat_features, axis=1), dummies.astype('int64')], axis=1)

### **Feature Engineering**

As there is no scaling performed, no data leakage -> only clear categories, which are supposed to be known beforehand.

In [16]:
df.loc[:, df.nunique() > 2].head()

,Timestamp,Packet Length,Anomaly Scores,Device Information,Geo-location Data
0,2023-05-30 06:33:58,503,28.67,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,"Jamshedpur, Sikkim"
1,2020-08-26 07:08:30,1174,51.50,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,"Bilaspur, Nagaland"
2,2022-11-13 08:23:25,306,87.42,Mozilla/5.0 (compatible; MSIE 9.0; Windows NT ...,"Bokaro, Rajasthan"
3,2023-07-02 10:38:46,385,15.79,Mozilla/5.0 (Macintosh; PPC Mac OS X 10_11_5; ...,"Jaunpur, Rajasthan"
4,2023-07-16 13:11:07,1462,0.52,Mozilla/5.0 (compatible; MSIE 5.0; Windows NT ...,"Anantapur, Tripura"


As discussed in EDA, those features, derived from Device Information hold the most info.

In [17]:
df['Browser'] = df['Device Information'].str.split('/').str[0] == 'Mozilla'
df['Browser'] = df['Browser'].astype(int)

In [18]:
df['Windows OS'] = df['Device Information'].str.contains('Windows').astype(int)
df['iOS'] = df['Device Information'].str.contains('iPhone|Mac').astype(int)
df['Linux OS'] = df['Device Information'].str.contains('Linux|Android').astype(int)

In [19]:
df.drop('Device Information', axis=1, inplace=True)

Also, the timestamp might provide some information.

In [20]:
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df['hour'] = df['Timestamp'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

sin, cos are used to represent the cyclical nature of the time - Monday, ..., Sunday -> Monday e.t.c.

The timestamp is dropped as it is causing data leakage.

In [21]:
df.drop('Timestamp', axis=1, inplace=True)

There are 28 states -> a good feature for target encoding: too big for one-hot encoding, but still has ~ 1.5k samples per State.

In [22]:
df['State'] = df['Geo-location Data'].str.split(', ').str[1]

In [23]:
df.drop('Geo-location Data', axis=1, inplace=True)

## **Export Results**

In [24]:
df.to_csv('data_clean.csv', index=False)

Other feature engineerging stuff will be performed after the data split in another notebook!